In [1]:
import os
import urllib.request
from ultralytics import YOLO
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

model = YOLO("yolo11n.pt")
print(f"\n모델 로드 완료: yolo11n.pt")

os.makedirs("outputs", exist_ok=True)
img_path = "outputs/bus.jpg"
urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", img_path)
results = model(img_path)

for result in results:
    boxes = result.boxes
    print(f"검출된 객체 수: {len(boxes)}")

    for i, box in enumerate(boxes):
        cls_id = int(box.cls)
        cls_name = model.names[cls_id]
        conf = box.conf.item()
        xyxy = box.xyxy[0].tolist()

        print(f"객체 {i}: {cls_name} (conf={conf:.3f})")
        print(f"BBox: [{xyxy[0]:.1f}, {xyxy[1]:.1f},"
              f"{xyxy[2]:.1f}, {xyxy[3]:.1f}]")
        
    annotated = result.plot()
    cv2.imwrite("outputs/inference_result.jpg", annotated)
    print("\n결과 저장: outputs/inference_result.jpg")

for conf_thresh in [0.25, 0.50, 0.75]:
    results = model(img_path,
                    conf=conf_thresh, verbose=False)
    n_detections = len(results[0].boxes)
    print(f"conf={conf_thresh:.2f}: {n_detections}개 검출")

WARNING ⚠️ user config directory '/root/.config/Ultralytics' is not writable, using '/tmp/Ultralytics'. Set YOLO_CONFIG_DIR to override.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

모델 로드 완료: yolo11n.pt

Found https://ultralyics.com/images/bus.jpg locally at bus.jpg
image 1/1 /workspace/study/physical-ai-study/Studies/Phase 3/week4/bus.jpg: 640x480 4 persons, 1 bus, 46.9ms
Speed: 2.4ms preprocess, 46.9ms inference, 10.7ms postprocess per image at shape (1, 3, 640, 480)
검출된 객체 수: 5
객체 0: bus (conf=0.940)
BBox: [3.8, 229.4,796.2, 728.4]
객체 1: person (conf=0.888)
BBox: [671.0, 394.8,809.8, 878.7]
객체 2: person (conf=0.878)
BBox: [47.4, 399.6,239.3, 904.2]
객체 3: person (conf=0.856)
BBox: [223.1, 408.7,344.5, 860.4]
객체 4: person (conf=0.6

In [ ]:
from ultralytics import YOLO
from ultralytics.utils import RUNS_DIR
import os

model = YOLO("yolo11n.pt")

# RUNS_DIR은 ultralytics 기본 저장 위치(physical-ai-study/runs)다. 그 부모인
# physical-ai-study를 기준으로 week4 경로를 조립하면 /workspace/study 같은 머신
# 종속 경로를 하드코딩하지 않아도 된다. 절대 경로라 runs/detect 중첩도 없다
project_dir = str(RUNS_DIR.parent / "Studies/Phase 3/week4/outputs/runs/detect")

results = model.train(
    data="coco128.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project=project_dir,
    name="coco128_baseline",
    patience=10,
    save=True,
    plots=True,
    verbose=True,
)

result_dir = f"{project_dir}/coco128_baseline"

if os.path.exists(result_dir):
    files = os.listdir(result_dir)
    print(f"결과 디렉토리: {result_dir}")
    print(f"생성된 파일: {files}")

best_model = YOLO(f"{result_dir}/weights/best.pt")
metrics = best_model.val()

print(f"\n mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")


print("\n결과 파일 확인:")
print(f"- {result_dir}/results.png (학습 커브)")
print(f"- {result_dir}/confusion_matrix.png (혼동 행렬)")
print(f"- {result_dir}/PR_curve.png (PR 커브)")